# Week 2 Advanced — Flexible Drivetrain, Resonance, and Controller Interaction

Now the controller can accidentally excite a torsional mode. The plant is

$$J_1\dot{\omega}_1=T_m-k\phi-c(\omega_1-\omega_2)$$
$$J_2\dot{\omega}_2=k\phi+c(\omega_1-\omega_2)-T_L$$
$$\dot{\phi}=\omega_1-\omega_2.$$

The engineering problem is no longer just 'track speed'; it is 'track speed without pumping energy into the shaft mode.'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import StateSpace, bode

J1, J2, k, c = 0.05, 0.09, 10.0, 0.08
A = np.array([[-c/J1, c/J1, -k/J1], [c/J2, -c/J2, k/J2], [1, -1, 0]])
B = np.array([[1/J1], [0], [0]])
C = np.array([[1,0,0], [0,1,0], [0,0,1]])
print('Plant poles:', np.linalg.eigvals(A))


## 1. Find the resonance before tuning the controller

The motor-side speed transfer function can have a resonance/antiresonance structure. Sweep frequency and identify the flexible mode.

In [ ]:
sys = StateSpace(A, B, np.array([[1,0,0]]), np.array([[0.0]]))
w = np.logspace(-1, 2.5, 600)
w, mag, phase = bode(sys, w=w)
plt.figure(figsize=(9,4))
plt.semilogx(w, mag)
plt.xlabel('Frequency [rad/s]'); plt.ylabel('Magnitude [dB]'); plt.grid(True, which='both'); plt.show()
print('Peak near [rad/s]:', w[np.argmax(mag)])


## 2. Close a speed loop and deliberately make it too aggressive

Compare a conservative PI loop against one whose bandwidth pushes into the flexible mode.

In [ ]:
def run(kp, ki):
    dt=2e-4; tt=np.arange(0,3,dt); x=np.zeros(3); integ=0.0
    H=[]
    for ti in tt:
        ref=25.0; load=0.3 if ti<1.5 else 0.8
        e=ref-x[0]; integ += e*dt
        u=np.clip(kp*e+ki*integ,-5,5)
        dx=A@x+B[:,0]*u+np.array([0,-load/J2,0])
        x=x+dx*dt
        H.append([*x,u])
    return tt,np.array(H)

t1,h1=run(0.18,0.7)
t2,h2=run(0.9,5.0)
plt.figure(figsize=(9,4))
plt.plot(t1,h1[:,0],label='conservative motor speed')
plt.plot(t2,h2[:,0],label='aggressive motor speed')
plt.plot(t2,h2[:,1],ls='--',label='aggressive load speed')
plt.grid(True); plt.legend(); plt.xlabel('Time [s]'); plt.ylabel('rad/s'); plt.show()


## Engineering tasks

- Compute the flexible-mode natural frequency and damping from the complex poles.
- Explain why measuring only motor-side speed can hide shaft stress.
- Add a shaft-twist limit and redesign the speed loop.
- Increase shaft stiffness by 2x and predict the resonance shift before running.
- Add a notch filter at the torsional frequency and compare.

**Deliverable:** show a controller that meets speed tracking while keeping peak $|\phi|$ below a chosen mechanical limit.